### RAG Pipelines-Data Ingestion to Vector DB

In [30]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [31]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: model.pdf
  ✓ Loaded 12 pages

Processing: nickel.pdf
  ✓ Loaded 8 pages

Total documents loaded: 20


In [32]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [33]:
chunks=split_documents(all_pdf_documents)
chunks

Split 20 documents into 78 chunks

Example chunk:
Content: Jurnal Sains dan Informatika 
 
p-ISSN: 2460-173X 
Volume 9, Nomor 1, Juni 2023 
 
e-ISSN: 2598-5841 
 
 
DOI: 10.34128/jsi.v9i1.523 
Received: 11 November 2022 
 
Accepted: 27 Juni 2023 
 
77 
Pemode...
Metadata: {'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': '', 'creationdate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'file_path': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20230811022238Z00'00'", 'trapped': '', 'modDate': "D:20230811022238Z00'00'", 'creationDate': "D:20230811022238Z00'00'", 'page': 0, 'source_file': 'model.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': '', 'creationdate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'file_path': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20230811022238Z00'00'", 'trapped': '', 'modDate': "D:20230811022238Z00'00'", 'creationDate': "D:20230811022238Z00'00'", 'page': 0, 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content='Jurnal Sains dan Informatika \n \np-ISSN: 2460-173X \nVolume 9, Nomor 1, Juni 2023 \n \ne-ISSN: 2598-5841 \n \n \nDOI: 10.34128/jsi.v9i1.523 \nReceived: 11 November 2022 \n \nAccepted: 27 Juni 2023 \n \n77 \nPemodelan Sistem Deteksi Kadar Unsur Hara Tanah Berdasarkan \nNilai NPK Menggunakan Metode Fuzzy Mamdani \n \nDityo Kreshna Argeshwara1), Zulkham Umar Rosyidin2), Aji Prasetya Wibawa3),  \nAnik Nur Handayani4), Mokh. Sholihul Hadi5) \n \n12345) Universitas Nege

## embeding And VectorStoreDB

In [34]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
class EmbeddingManager:
  """Handle document embedding generation using SentenceTransformer"""
  def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
    """
    initiate the embedding manager

    Args:
        model_name : HuggingFace model name for sentence embedding
    """
    self.model_name = model_name
    self.model = None
    self._load_model()

  def _load_model(self):
    """Load the SentenceTransformer model"""
    try:
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
    except Exception as e:
        print(f"Error loading model {self.model_name}: {e}")
        raise
        
  def generate_embeddings(self, texts: List[str]) -> np.ndarray:
    """
    Generate embedding for a list of texts
    
    Args:
        texts: List of strings to embed

    Return: 
    numpy array of shape (len(texts), embedding_dim)
    """
    if not self.model:
      raise ValueError("Model not loaded")

    print(f"Generating embeddings for {len(texts)} texts...")
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f"✓ Generated embeddings with shape: {embeddings.shape}")
    return embeddings
  
## innitiate the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6977.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15228\2193690588.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStrore

In [36]:
class VectorStore:
  """
  Manages documents embeddings in chromadb vector store
  """
  def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
    """
    Initialize the vector store

    Args:
        collection_name: Name of the chromadb collection
        persist_directory: Directory to persist the chromadb database
    """
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
    """initialize Chromadb client and collection"""
    try:
      #create persist directory if not exists
      os.makedirs(self.persist_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)

      #get or create collection
      self.collection = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata={"description": "PDF document embeddings for RAG system"}
      )
      print(f"Vector store initialized. Collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}")
    
    except Exception as e:
      print(f"✗ Error initializing Chromadb: {e}")
      raise e
    
  def add_documents(self, documents: List[Any], embeddings: np.ndarray):
      """
      Add documents and their embeddings to the vector store

      Args:
          documents: List of document objects (e.g., Langchain Document)
          embeddings: Numpy array of shape (len(documents), embedding_dim)
      """
      if len(documents) != len(embeddings):
        raise ValueError("Number of documents and embeddings must match")
      
      print(f"Adding {len(documents)} documents to vector store...")

      #prepare data for chromadb
      ids = []
      metadatas = []
      documents_text = []
      embeddings_list = []

      for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
        #generate unique id for each document
        doc_id = str(uuid.uuid4())
        ids.append(doc_id)

        #prepare metadata
        metadata = dict(doc.metadata)  #copy metadata from document
        metadata['doc_index'] = i
        metadata['content_length'] = len(doc.page_content)
        metadatas.append(metadata)

        #document content
        documents_text.append(doc.page_content)
        
        #embedding
        embeddings_list.append(embedding.tolist())  #convert numpy array to list for chromadb

      #add to chromadb collection
      try:
        self.collection.add(
          ids=ids,
          metadatas=metadatas,
          documents=documents_text,
          embeddings=embeddings_list
        )
        print(f"Successfully added {len(documents)} documents to vector store")
        print(f"Total documents in collection: {self.collection.count()}")
        
      except Exception as e:
        print(f"✗ Error adding documents to vector store: {e}")
        raise e
      
vector_store = VectorStore()
vector_store




  

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 457


In [37]:
chunks

[Document(metadata={'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': '', 'creationdate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'file_path': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20230811022238Z00'00'", 'trapped': '', 'modDate': "D:20230811022238Z00'00'", 'creationDate': "D:20230811022238Z00'00'", 'page': 0, 'source_file': 'model.pdf', 'file_type': 'pdf'}, page_content='Jurnal Sains dan Informatika \n \np-ISSN: 2460-173X \nVolume 9, Nomor 1, Juni 2023 \n \ne-ISSN: 2598-5841 \n \n \nDOI: 10.34128/jsi.v9i1.523 \nReceived: 11 November 2022 \n \nAccepted: 27 Juni 2023 \n \n77 \nPemodelan Sistem Deteksi Kadar Unsur Hara Tanah Berdasarkan \nNilai NPK Menggunakan Metode Fuzzy Mamdani \n \nDityo Kreshna Argeshwara1), Zulkham Umar Rosyidin2), Aji Prasetya Wibawa3),  \nAnik Nur Handayani4), Mokh. Sholihul Hadi5) \n \n12345) Universitas Nege

In [38]:
### convert the chunks into text for embedding
texts = [doc.page_content for doc in chunks]

#generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

#store in vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 78 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.18s/it]

✓ Generated embeddings with shape: (78, 384)
Adding 78 documents to vector store...
Successfully added 78 documents to vector store
Total documents in collection: 535


### Retriever Pipeline From VectorStore

In [39]:
class RAGRetriever:
  """Handles query-based retrieval from the vector store"""

  def __init__(self,vector_store: VectorStore, embedding_manager: EmbeddingManager):
    """
    initialize the retriever
    
    kwargs:
      vector_store: vector store containing document embeddings
      embedding_manager: managger for generating query embeddings
    """
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager

  def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
    """
    Retrieve relevant documents for a given query

    Args:
        query: User query string
        top_k: Number of top relevant documents to retrieve
        score_threshold: Minimum cosine similarity score for retrieved documents
    
    Returns:
        List of retrieved documents with metadata and similarity scores
    """
    print(f"retrieving documents for query: '{query}'")
    print(f"Top K: {top_k}, Score_threshold: {score_threshold}")

    #generate query embedding
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    #search in vector store
    try:
      results = self.vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
      )

      retrived_docs = []

      if results['documents'] and results['documents'][0]:
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]

        for i, (doc_id, document, metadata, distance) in enumerate(zip(ids,documents,metadatas,distances)):
          #convert distance to similarity score (chromeDB using cosine distance)
          # Cosine similarity = 1 - cosine_distance
          similarity_score = 1 - distance

          if similarity_score >= score_threshold:
            retrived_docs.append({
              "id":doc_id,
              "content":document,
              "metadata":metadata,
              "similarity_score":similatity_scores,
              "distance":distance,
              "rank":i+1
              })
        print(f"retrieved {len(retrived_docs)} documents (after Filtering)")
      else:
        print("No document found")

      return retrived_docs

    except Exception as e:
      print(f'Error dirung retrieval: {e}')
      return []   

rag_retriever = RAGRetriever(vector_store,embedding_manager)
   
    

In [40]:
rag_retriever

In [53]:
rag_retriever.retrieve("Jurnal Sains dan Informatika ")

retrieving documents for query: 'Jurnal Sains dan Informatika '
Top K: 5, Score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.83it/s]

✓ Generated embeddings with shape: (1, 384)
Error dirung retrieval: name 'similatity_scores' is not defined


[]

In [52]:
rag_retriever.retrieve("Python")

retrieving documents for query: 'Python'
Top K: 5, Score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.24it/s]

✓ Generated embeddings with shape: (1, 384)
retrieved 0 documents (after Filtering)


[]

In [43]:
# Debug: Check raw results without filtering
query = "Python programming language"
query_embedding = embedding_manager.generate_embeddings([query])[0]

results = vector_store.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

print("Raw results from vector store:")
print(f"IDs: {results['ids']}")
print(f"Distances: {results['distances']}")
if results['documents'] and results['documents'][0]:
    print(f"Number of documents returned: {len(results['documents'][0])}")
    for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
        similarity = -dist
        print(f"\nResult {i+1}:")
        print(f"  Similarity: {similarity:.4f}")
        print(f"  Distance: {dist:.4f}")
        print(f"  Content: {doc[:100]}...")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 69.02it/s]

✓ Generated embeddings with shape: (1, 384)
Raw results from vector store:
IDs: [['6402a7fa-cce3-4255-8fbe-00d6c103dffe', 'efc4f518-bcd8-4e52-9e8c-d862c71210ef', '42a40195-f772-4731-ba8d-3f363d26b1e3', '99d897f7-7c08-4202-8412-3a717319119a', 'c78cbff3-4869-464f-8dc9-aac6aa824431']]
Distances: [[1.4798496961593628, 1.5051177740097046, 1.538919448852539, 1.5569490194320679, 1.5818603038787842]]
Number of documents returned: 5

Result 1:
  Similarity: -1.4798
  Distance: 1.4798
  Content: best models from the literature. We show that the Transformer generalizes well to
other tasks by app...

Result 2:
  Similarity: -1.5051
  Distance: 1.5051
  Content: across languages. In Proceedings of the 2009 Conference on Empirical Methods in Natural
Language Pro...

Result 3:
  Similarity: -1.5389
  Distance: 1.5389
  Content: In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relyi...

Result 4:
  Similarity: -1.5569
  Distance: 1.5569
  Content: 1 Introd

In [44]:
# Fixed retrieval function
def retrieve_fixed(query: str, top_k: int = 5, score_threshold: float = 0.3):
    """Retrieve with correct similarity calculation"""
    query_embedding = embedding_manager.generate_embeddings([query])[0]
    
    results = vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    retrieved = []
    if results['documents'] and results['documents'][0]:
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]
        ids = results['ids'][0]
        
        for doc_id, document, metadata, distance in zip(ids, documents, metadatas, distances):
            # CORRECT: Cosine similarity = 1 - cosine_distance
            similarity_score = 1 - distance
            
            if similarity_score >= score_threshold:
                retrieved.append({
                    "id": doc_id,
                    "content": document,
                    "metadata": metadata,
                    "similarity_score": similarity_score,
                    "distance": distance
                })
    
    print(f"Query: '{query}'")
    print(f"Retrieved {len(retrieved)} documents (threshold: {score_threshold})")
    return retrieved

# Test dengan berbagai queries
results = retrieve_fixed("Python", score_threshold=0.1)
for i, doc in enumerate(results[:3]):
    print(f"\n[{i+1}] Similarity: {doc['similarity_score']:.4f}")
    print(f"    Content: {doc['content'][:80]}...")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.80it/s]

✓ Generated embeddings with shape: (1, 384)
Query: 'Python'
Retrieved 0 documents (threshold: 0.1)


In [45]:
# Check top similarities without threshold
def check_similarities(query: str, top_k: int = 5):
    query_embedding = embedding_manager.generate_embeddings([query])[0]
    results = vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    print(f"Query: '{query}'")
    print(f"Top {top_k} results:")
    if results['documents'] and results['documents'][0]:
        for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
            similarity = 1 - dist
            print(f"  [{i+1}] Similarity: {similarity:.4f} | Content: {doc[:60]}...")
    return results

check_similarities("Transformer model", top_k=5)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.24it/s]

✓ Generated embeddings with shape: (1, 384)
Query: 'Transformer model'
Top 5 results:
  [1] Similarity: -0.1668 | Content: Figure 1: The Transformer - model architecture.
The Transfor...
  [2] Similarity: -0.1722 | Content: Table 2: The Transformer achieves better BLEU scores than pr...
  [3] Similarity: -0.2829 | Content: The Transformer uses multi-head attention in three different...
  [4] Similarity: -0.2836 | Content: Table 3: Variations on the Transformer architecture. Unliste...
  [5] Similarity: -0.2862 | Content: inference to input length + 50, but terminate early when pos...


{'ids': [['d6564664-88a4-4a14-9b1e-e773f5e8528e',
   'a61a4f1d-a93d-48b8-bb4a-b987e3db6ae3',
   '1e7fd7d6-f36d-49c6-acb8-549d7c4eb77e',
   '66169a51-e54c-471c-82dd-42efd5ddf045',
   'aa4a806a-6b3b-48f5-a6d0-091a7679421b']],
 'embeddings': None,
 'documents': [['Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-\nwise fully connected feed-forward network. We employ a residual connection [11] around each of\nthe two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is\nLayerNorm(x + Sublayer(x)), where Sublayer(x) is t

In [46]:
# SOLUTION: ChromaDB has built-in all-MiniLM-L6-v2 (same as ours!)
# The issue: We're mixing embeddings from different sources
# FIX: Use ChromaDB's automatic embedding with query_texts

# Create collection with different name to avoid lock
class VectorStoreFresh:
    def __init__(self, collection_name: str = "pdf_documents_v2"):
        self.collection_name = collection_name
        self.persist_directory = "../data/vector_store"
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF documents with ChromaDB default embedding"}
        )
        print(f"✓ Initialized collection: {self.collection_name}")
        print(f"  Existing docs: {self.collection.count()}")

# Create fresh store
store_fresh = VectorStoreFresh()

# Add documents (let ChromaDB auto-embed!)
ids = []
metadatas = []
documents_text = []

for i, doc in enumerate(chunks):
    doc_id = str(uuid.uuid4())
    ids.append(doc_id)
    
    metadata = dict(doc.metadata)
    metadata['doc_index'] = i
    metadata['content_length'] = len(doc.page_content)
    metadatas.append(metadata)
    
    documents_text.append(doc.page_content)

try:
    store_fresh.collection.add(
        ids=ids,
        metadatas=metadatas,
        documents=documents_text
        # NO embeddings - ChromaDB auto-generates using default all-MiniLM-L6-v2
    )
    print(f"✓ Added {len(chunks)} documents")
    print(f"✓ Total in collection: {store_fresh.collection.count()}")
except Exception as e:
    print(f"✗ Error: {e}")

# Test retrieval with query_texts (auto-embedding)
test_query = "Transformer attention mechanism"
results = store_fresh.collection.query(
    query_texts=[test_query],  # Auto-embed this text!
    n_results=3
)

print(f"\nQuery: '{test_query}'")
if results['documents'] and results['documents'][0]:
    for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
        similarity = 1 - dist
        print(f"  [{i+1}] Similarity: {similarity:.4f} | {doc[:70]}...")

✓ Initialized collection: pdf_documents_v2
  Existing docs: 78
✓ Added 78 documents
✓ Total in collection: 156

Query: 'Transformer attention mechanism'
  [1] Similarity: -0.5826 | [13] K. 
H. 
Suradiradja, 
“Algoritme 
Machine 
Learning 
Multi-Layer ...
  [2] Similarity: -0.5826 | [13] K. 
H. 
Suradiradja, 
“Algoritme 
Machine 
Learning 
Multi-Layer ...
  [3] Similarity: -0.6272 | dasar RNN adalah menggunakan algoritma yang 
dilakukan berulang, yang ...


In [47]:
# Debug: Check what's in chunks
print(f"Total chunks: {len(chunks)}")
print(f"\nFirst 3 chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[{i+1}] Metadata: {chunk.metadata}")
    print(f"    Content: {chunk.page_content[:100]}...")

# Check if chunks contain non-PDF content
pdf_chunks = [c for c in chunks if c.metadata.get('file_type') == 'pdf']
print(f"\nChunks from PDFs: {len(pdf_chunks)}")
print(f"Chunks from other sources: {len(chunks) - len(pdf_chunks)}")

Total chunks: 78

First 3 chunks:

[1] Metadata: {'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': '', 'creationdate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'file_path': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20230811022238Z00'00'", 'trapped': '', 'modDate': "D:20230811022238Z00'00'", 'creationDate': "D:20230811022238Z00'00'", 'page': 0, 'source_file': 'model.pdf', 'file_type': 'pdf'}
    Content: Jurnal Sains dan Informatika 
 
p-ISSN: 2460-173X 
Volume 9, Nomor 1, Juni 2023 
 
e-ISSN: 2598-5841...

[2] Metadata: {'producer': 'macOS Version 13.2.1 (Build 22D68) Quartz PDFContext', 'creator': '', 'creationdate': "D:20230811022238Z00'00'", 'source': '..\\data\\pdf\\model.pdf', 'file_path': '..\\data\\pdf\\model.pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:2023

In [48]:
# Simpler debug
print(f"Chunks count: {len(chunks)}")
print(f"Chunk[0] source: {chunks[0].metadata.get('source_file')}")

# Get unique sources
sources = set(c.metadata.get('source_file', 'Unknown') for c in chunks)
print(f"Sources in chunks: {sources}")

# Raw distance check
q_result = store_fresh.collection.query(query_texts=["machine"], n_results=1)
if q_result['documents'] and q_result['documents'][0]:
    dist = q_result['distances'][0][0]
    print(f"\nDistance value: {dist:.4f}")
    print(f"Similarity (1-d): {1-dist:.4f}")
    print(f"Similarity (-d): {-dist:.4f}")

Chunks count: 78
Chunk[0] source: model.pdf
Sources in chunks: {'model.pdf', 'nickel.pdf'}

Distance value: 1.5716
Similarity (1-d): -0.5716
Similarity (-d): -1.5716


In [49]:
# Check ChromaDB embedding function
print(f"Collection metadata: {store_fresh.collection.metadata}")

# Try to see embeddings directly
all_data = store_fresh.collection.get(include=["embeddings"])
embeddings_list = all_data.get('embeddings')

if embeddings_list is not None and len(embeddings_list) > 0:
    first_emb = embeddings_list[0]
    first_emb_arr = np.array(first_emb) if not isinstance(first_emb, np.ndarray) else first_emb
    
    print(f"First document embedding dimension: {len(first_emb_arr)}")
    print(f"Embedding sample: {first_emb_arr[:5]}...")  # First 5 values
    print(f"Embedding L2 norm: {np.linalg.norm(first_emb_arr):.4f}")
    
    # Get corresponding text
    test_doc = chunks[0].page_content[:200]
    manual_emb = embedding_manager.generate_embeddings([test_doc])[0]
    print(f"\nManual embedding dimension: {len(manual_emb)}")
    print(f"Manual embedding sample: {manual_emb[:5]}...")
    print(f"Manual embedding L2 norm: {np.linalg.norm(manual_emb):.4f}")
    
    # Compare
    cosine_sim = np.dot(first_emb_arr, manual_emb) / (np.linalg.norm(first_emb_arr) * np.linalg.norm(manual_emb))
    print(f"\nCosine similarity between ChromaDB and SentenceTransformer: {cosine_sim:.4f}")

Collection metadata: {'description': 'PDF documents with ChromaDB default embedding'}
First document embedding dimension: 384
Embedding sample: [ 0.01303509 -0.02103169  0.02542459  0.0023853  -0.04059899]...
Embedding L2 norm: 1.0000
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.51it/s]

✓ Generated embeddings with shape: (1, 384)

Manual embedding dimension: 384
Manual embedding sample: [-0.02499034  0.11973957 -0.04089223 -0.01213403 -0.05202983]...
Manual embedding L2 norm: 1.0000

Cosine similarity between ChromaDB and SentenceTransformer: 0.6134


In [50]:
# FINAL SOLUTION: Use ChromaDB's ONNX embeddings exclusively
# Recreate RAG retriever using ChromaDB default embedding

class RAGRetrieverFixed:
    """Retriever using ChromaDB's built-in embedding function"""
    def __init__(self, vector_store, use_auto_embedding=True):
        self.vector_store = vector_store
        self.use_auto_embedding = use_auto_embedding
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = -0.3):
        """Retrieve documents using ChromaDB's built-in embedding"""
        try:
            # Use query_texts for automatic embedding
            results = self.vector_store.collection.query(
                query_texts=[query],
                n_results=top_k
            )
            
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for doc_id, document, metadata, distance in zip(ids, documents, metadatas, distances):
                    # Cosine similarity = 1 - cosine_distance
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance
                        })
            
            print(f"Query: '{query}'")
            print(f"Retrieved {len(retrieved_docs)} documents")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error: {e}")
            return []

# Create retriever with fresh vector store
rag_retriever_fixed = RAGRetrieverFixed(store_fresh)

# Test with different queries
print("=" * 60)
test_queries = [
    "Transformer model architecture",
    "attention mechanism",
    "machine learning",
]

for query in test_queries:
    docs = rag_retriever_fixed.retrieve(query, top_k=2, score_threshold=-1.0)  # Accept all
    print(f"  Top result similarity: {docs[0]['similarity_score']:.4f}" if docs else "  No results")
    print()

Query: 'Transformer model architecture'
Retrieved 2 documents
  Top result similarity: -0.4256

Query: 'attention mechanism'
Retrieved 2 documents
  Top result similarity: -0.4611

Query: 'machine learning'
Retrieved 2 documents
  Top result similarity: -0.2062



In [51]:
# Test dengan threshold yang lebih baik
print("\n" + "=" * 60)
print("FINAL TEST - Dengan threshold optimized:")
print("=" * 60)

result = rag_retriever_fixed.retrieve("Jurnal Sains dan Informatika", top_k=5, score_threshold=-0.6)
print(f"\nTotal retrieved: {len(result)} documents")
if result:
    for i, doc in enumerate(result[:3], 1):
        print(f"\n[{i}] Similarity: {doc['similarity_score']:.4f}")
        print(f"    Source: {doc['metadata'].get('source_file', 'Unknown')}")
        print(f"    Content: {doc['content'][:100]}...")


FINAL TEST - Dengan threshold optimized:
Query: 'Jurnal Sains dan Informatika'
Retrieved 5 documents

Total retrieved: 5 documents

[1] Similarity: -0.0851
    Source: model.pdf
    Content: Jurnal Sains dan Informatika 
 
p-ISSN: 2460-173X 
Volume 9, Nomor 1, Juni 2023 
 
e-ISSN: 2598-5841...

[2] Similarity: -0.0851
    Source: model.pdf
    Content: Jurnal Sains dan Informatika 
 
p-ISSN: 2460-173X 
Volume 9, Nomor 1, Juni 2023 
 
e-ISSN: 2598-5841...

[3] Similarity: -0.1285
    Source: model.pdf
    Content: Jurnal Sains dan Informatika 
 
p-ISSN: 2460-173X 
Volume X, Nomor X, Juni/November 2020 
 
e-ISSN: ...
